In [1]:
!pip install -q transformers accelerate bitsandbytes sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.7 MB/s eta 0:00:00:00:0100:01


In [2]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm import tqdm
import os
from google.colab import drive
import gc

In [3]:
# Free up memory
if 'model' in locals():
    del model
if 'tokenizer' in locals():
    del tokenizer

gc.collect()
torch.cuda.empty_cache()

In [4]:
# 1. Setup Drive and Config
drive.mount('/content/drive')
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

Mounted at /content/drive


In [5]:
input_path = '/content/drive/MyDrive/Project/results/baseline_results.csv'
df = pd.read_csv(input_path)

In [6]:
df.head(2)

,index,code,cwe,cve,truth_description,baseline_explanation
0,0,"ExprResolveLhs(struct xkb_context *ctx, const ...",CWE-476,CVE-2018-15859,Fail expression lookup on invalid atoms\n\nIf ...,The vulnerability in this code is that it does...
1,1,OperationID FileSystemOperationRunner::BeginOp...,CWE-190,CVE-2019-5788,[FileSystem] Harden against overflows of Opera...,The vulnerability in this C code is that it do...


In [7]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [ ]:
def run_verification(model_id="deepseek-ai/deepseek-coder-7b-instruct-v1.5"):
    print(f"--- Loading Judge Model: {model_id} ---")
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        low_cpu_mem_usage=True,  # Helps reduce the memory spike during loading
        offload_folder="offload" # Provides a safety buffer if memory is tight
    )

    # Load results from the previous inference
    input_path = '/content/drive/MyDrive/Project/results/baseline_results.csv'
    df = pd.read_csv(input_path)
    
    verified_results = []

    print("--- Starting Verification Process ---")
    for index, row in tqdm(df.iterrows(), total=len(df)):
        # 1. Structure the Ground Truth Profile from specific columns
        # forces the Judge to consider the Category (CWE), ID (CVE), and Developer Intent (Commit Message)
        ground_truth_block = f"""
        - Vulnerability Type: {row.get('cwe', 'Unknown')}
        - Reference ID: {row.get('cve', 'Unknown')}
        - Developer Commit Message (Root Cause): {row.get('truth_description', 'No description available')}
        """

        # 2. Construct the Judge Prompt with high-strictness instructions
        judge_prompt = f"""
        [INST] You are a Senior Security Auditor. 
        Verify if the AI's explanation correctly identifies the specific logical root cause described in the GROUND TRUTH.
        
        CODE: 
        {row['code']}

        GROUND TRUTH: 
        {ground_truth_block}

        AI EXPLANATION: 
        {row['baseline_explanation']}
        
        CRITERIA:
        1. Does the AI identify the correct variable/function failure mentioned in the Commit Message?
        2. Does the AI match the vulnerability type (CWE)?
        3. Ignore general "best practice" advice (like missing null checks on arguments) if it is NOT the core issue in the Commit Message.

        Does the AI EXPLANATION accurately identify the specific vulnerability? 
        Answer ONLY 'YES' or 'NO' followed by a one-sentence reason. [/INST]
        """
        
        # Use absolute max context (4096) to fit code + truth + explanation
        inputs = tokenizer(judge_prompt, return_tensors="pt", truncation=True, max_length=4096).to("cuda")
        
        with torch.no_grad():
            # Greedy decoding (temp=0.1) for objective classification
            outputs = model.generate(**inputs, max_new_tokens=100, temperature=0.1)
        
        judge_response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
        
        # Determine correctness (check the first word of response)
        is_correct = "YES" in judge_response.upper()[:10]

        verified_results.append({
            **row.to_dict(),
            'judge_response': judge_response,
            'is_correct': is_correct
        })

    # Save the filtered results
    output_df = pd.DataFrame(verified_results)
    output_path = '/content/drive/MyDrive/Project/results/verified_baseline.csv'
    output_df.to_csv(output_path, index=False)
    
    # Summary for the Researcher
    pass_count = output_df['is_correct'].sum()
    print(f"--- Verification Complete! {pass_count}/{len(df)} samples passed. ---")
    print(f"Verified data saved to {output_path}")

In [9]:
if __name__ == "__main__":
    run_verification()

--- Loading Judge Model: deepseek-ai/deepseek-coder-7b-instruct-v1.5 ---


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/621 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/273 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

--- Starting Verification Process ---


  0%|          | 0/150 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:100015 for open-end generation.
  6%|▌         | 9/150 [01:32<22:26,  9.55s/it]Setting `pad_token_id` to `eos_token_id`:100015 for open-end generation.
This is a friendly reminder - the current text generation call has exceeded the model's predefined maximum length (4096). Depending on the model, you may observe exceptions, performance degradation, or nothing at all.
100%|██████████| 150/150 [37:17<00:00, 14.92s/it]

--- Verification Complete! 64/150 samples passed. ---
Verified data saved to /content/drive/MyDrive/Project/results/verified_baseline.csv


In [10]:
input_path = '/content/drive/MyDrive/Project/results/verified_baseline.csv'
df = pd.read_csv(input_path)

# Check how many different types of bugs you have left
passed_df = df[df['is_correct'] == True]
print(passed_df['cwe'].value_counts())

cwe
CWE-119    12
CWE-20      9
CWE-264     7
CWE-399     5
CWE-125     5
CWE-190     4
CWE-362     4
CWE-189     3
CWE-476     2
CWE-416     2
CWE-494     1
CWE-285     1
CWE-59      1
CWE-665     1
CWE-732     1
CWE-79      1
CWE-254     1
CWE-787     1
CWE-19      1
CWE-200     1
Name: count, dtype: int64
